# Entity Extraction Evaluators Testing Notebook

This notebook demonstrates the usage of the modular evaluators for entity extraction quality assessment.

## Available Evaluators:
1. **ExtractionCorrectnessEvaluator**: Checks if extracted values are present in OCR text using fuzzy matching
2. **ExtractionCompletenessEvaluator**: Uses Azure OpenAI to assess if extracted values are relevant and complete

## Import Required Libraries

Import the evaluator classes and models from the evaluators package.

In [1]:
import sys
import os

# Add parent directory to path
sys.path.append(os.path.abspath('..'))

from evaluators.correctness_evaluator import ExtractionCorrectnessEvaluator
from evaluators.completeness_evaluator import ExtractionCompletenessEvaluator

print("✓ Evaluators imported successfully")

✓ Evaluators imported successfully


## 1. Extraction Correctness Evaluator

The **ExtractionCorrectnessEvaluator** uses fuzzy string matching (token set ratio) to determine if extracted values are present in the OCR text from Azure Document Intelligence.

### Key Features:
- Returns fuzzy similarity score (0-1)
- Marks extraction as correct only when fuzzy_score == 1.0
- Handles text normalization and cleaning automatically
- No threshold configuration needed

### Example 1.1: Basic Correctness Evaluation

Initialize the evaluator and test it with sample OCR text and extracted entities.

In [2]:
# Sample OCR text from Azure Document Intelligence
ocr_text = """
COMMERCIAL INVOICE

Invoice Number: INV-2024-00123
Invoice Date: January 15, 2024

Seller Information:
Company Name: Microsoft Corporation
Address: One Microsoft Way, Redmond, WA 98052
Tax ID: 91-1144442

Buyer Information:
Company: HSBC Bank
Contact: John Smith
Email: john.smith@hsbc.com

Line Items:
1. Software License - Enterprise    $25,000.00
2. Support Services - Annual        $5,000.00

Subtotal: $30,000.00
Tax (10%): $3,000.00
Total Amount Due: $33,000.00

Payment Terms: Net 30 Days
"""

# Initialize the correctness evaluator
correctness_eval = ExtractionCorrectnessEvaluator()

# Test with correctly extracted values
test_fields = [
    ("invoice_number", "INV-2024-00123"),
    ("seller_name", "Microsoft Corporation"),
    ("buyer_name", "HSBC Bank"),
    ("total_amount", "$33,000.00"),
    ("invoice_date", "January 15, 2024"),
]

print("=" * 80)
print("EXTRACTION CORRECTNESS EVALUATION RESULTS")
print("=" * 80)

for field_name, extracted_value in test_fields:
    result = correctness_eval.evaluate_field(
        field_name=field_name,
        extracted_value=extracted_value,
        source_text=ocr_text
    )
    
    status = "✓ CORRECT" if result.metadata['extraction_correct'] else "✗ INCORRECT"
    print(f"\nField: {field_name}")
    print(f"  Extracted Value: {extracted_value}")
    print(f"  Fuzzy Score: {result.metadata['fuzzy_score']:.2f}")
    print(f"  Status: {status}")

EXTRACTION CORRECTNESS EVALUATION RESULTS

Field: invoice_number
  Extracted Value: INV-2024-00123
  Fuzzy Score: 1.00
  Status: ✓ CORRECT

Field: seller_name
  Extracted Value: Microsoft Corporation
  Fuzzy Score: 1.00
  Status: ✓ CORRECT

Field: buyer_name
  Extracted Value: HSBC Bank
  Fuzzy Score: 1.00
  Status: ✓ CORRECT

Field: total_amount
  Extracted Value: $33,000.00
  Fuzzy Score: 1.00
  Status: ✓ CORRECT

Field: invoice_date
  Extracted Value: January 15, 2024
  Fuzzy Score: 1.00
  Status: ✓ CORRECT


### Example 1.2: Testing Incorrect Extractions

Test the evaluator with values that are NOT present in the OCR text to see how it handles incorrect extractions.

In [3]:
# Test with incorrect/missing values
incorrect_fields = [
    ("invoice_number", "INV-2024-99999"),  # Wrong number
    ("seller_name", "Apple Inc."),  # Wrong company
    ("total_amount", "$50,000.00"),  # Wrong amount
    ("payment_method", "Credit Card"),  # Not in document
]

print("\n" + "=" * 80)
print("TESTING INCORRECT EXTRACTIONS")
print("=" * 80)

for field_name, extracted_value in incorrect_fields:
    result = correctness_eval.evaluate_field(
        field_name=field_name,
        extracted_value=extracted_value,
        source_text=ocr_text
    )
    
    status = "✓ CORRECT" if result.metadata['extraction_correct'] else "✗ INCORRECT"
    print(f"\nField: {field_name}")
    print(f"  Extracted Value: {extracted_value}")
    print(f"  Fuzzy Score: {result.metadata['fuzzy_score']:.3f}")
    print(f"  Status: {status}")
    if 'reason' in result.metadata:
        print(f"  Reason: {result.metadata['reason']}")


TESTING INCORRECT EXTRACTIONS

Field: invoice_number
  Extracted Value: INV-2024-99999
  Fuzzy Score: 0.039
  Status: ✗ INCORRECT

Field: seller_name
  Extracted Value: Apple Inc.
  Fuzzy Score: 0.044
  Status: ✗ INCORRECT

Field: total_amount
  Extracted Value: $50,000.00
  Fuzzy Score: 0.044
  Status: ✗ INCORRECT

Field: payment_method
  Extracted Value: Credit Card
  Fuzzy Score: 0.044
  Status: ✗ INCORRECT


### Example 1.3: Batch Evaluation

Evaluate multiple fields at once using the `evaluate_batch` method for efficiency.

In [4]:
# Prepare batch data
batch_fields = [
    {"field_name": "invoice_number", "value": "INV-2024-00123"},
    {"field_name": "invoice_date", "value": "January 15, 2024"},
    {"field_name": "seller_name", "value": "Microsoft Corporation"},
    {"field_name": "buyer_name", "value": "HSBC Bank"},
    {"field_name": "subtotal", "value": "$30,000.00"},
    {"field_name": "tax", "value": "$3,000.00"},
    {"field_name": "total", "value": "$33,000.00"},
]

# Batch evaluate
results = correctness_eval.evaluate_batch(batch_fields, ocr_text)

print("\n" + "=" * 80)
print("BATCH EVALUATION RESULTS")
print("=" * 80)
print(f"\n{'Field Name':<20} {'Fuzzy Score':<15} {'Status':<15}")
print("-" * 80)

for result in results:
    status = "✓ CORRECT" if result.metadata['extraction_correct'] else "✗ INCORRECT"
    print(f"{result.field_name:<20} {result.metadata['fuzzy_score']:<15.3f} {status:<15}")


BATCH EVALUATION RESULTS

Field Name           Fuzzy Score     Status         
--------------------------------------------------------------------------------
invoice_number       1.000           ✓ CORRECT      
invoice_date         1.000           ✓ CORRECT      
seller_name          1.000           ✓ CORRECT      
buyer_name           1.000           ✓ CORRECT      
subtotal             1.000           ✓ CORRECT      
tax                  1.000           ✓ CORRECT      
total                1.000           ✓ CORRECT      


## 2. Extraction Completeness Evaluator

The **ExtractionCompletenessEvaluator** uses Azure OpenAI (GPT-4) to assess if extracted values are:
1. **Relevant** to the field name
2. **Complete** with all necessary information from the source

### Scoring Logic:
- **Relevance**: 50% of score (0.5 if relevant, 0.0 if not)
- **Completeness**: 50% of score
  - Complete: +0.5
  - 1 missing item: +0.3
  - 2 missing items: +0.2
  - 3+ missing items: +0.1

**Note**: This evaluator requires Azure OpenAI credentials and will make API calls.

### Example 2.1: Initialize Completeness Evaluator

Set up the Azure OpenAI client with your credentials. Update the endpoint and deployment name below.

In [5]:
# Azure OpenAI Configuration
import os
from dotenv import load_dotenv
load_dotenv()

AZURE_OPENAI_ENDPOINT = os.getenv("GPT_4_1_API_ENDPOINT")
DEPLOYMENT_NAME = os.getenv("GPT_4_1_API_DEPLOYMENT")
API_VERSION = os.getenv("GPT_4_1_API_VERSION")

# Initialize the completeness evaluator
# Uncomment when you have valid credentials
from azure.identity import DefaultAzureCredential

completeness_eval = ExtractionCompletenessEvaluator(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    deployment_name=DEPLOYMENT_NAME,
    api_version=API_VERSION,
    credential=DefaultAzureCredential()
)
print("✓ Completeness evaluator initialized")

✓ Completeness evaluator initialized


### Example 2.2: Evaluate Completeness (Demo Code)

This demonstrates how to use the completeness evaluator once Azure OpenAI is configured.

In [6]:
# Test different levels of completeness
test_cases = [
    ("seller_address", "One Microsoft Way, Redmond, WA 98052"),  # Complete
    ("seller_address", "Redmond, WA"),  # Incomplete - missing street
    ("invoice_amount", "$33,000"),  # Incomplete - missing cents
    ("invoice_amount", "$33,000.00"),  # Complete
    ("contact_info", "john.smith@hsbc.com"),  # Incomplete - missing name/phone
]

print("=" * 80)
print("COMPLETENESS EVALUATION RESULTS")
print("=" * 80)

for field_name, extracted_value in test_cases:
    result = completeness_eval.evaluate_field(
        field_name=field_name,
        extracted_value=extracted_value,
        source_text=ocr_text
    )
    
    print(f"\\nField: {field_name}")
    print(f"  Extracted Value: {extracted_value}")
    print(f"  Score: {result.score:.2f}")
    print(f"  Relevant: {result.metadata['is_relevant']}")
    print(f"  Complete: {result.metadata['is_complete']}")
    print(f"  Missing Info: {result.metadata['missing_info']}")
    print(f"  Reasoning: {result.metadata['reasoning']}")

COMPLETENESS EVALUATION RESULTS
\nField: seller_address
  Extracted Value: One Microsoft Way, Redmond, WA 98052
  Score: 0.80
  Relevant: True
  Complete: False
  Missing Info: ['Company Name: Microsoft Corporation']
  Reasoning: The extracted value 'One Microsoft Way, Redmond, WA 98052' is relevant as it is the seller's address. However, it is not complete because the source text also includes the company name 'Microsoft Corporation' as part of the seller's address block. The field should ideally include both the company name and the address for completeness.
\nField: seller_address
  Extracted Value: Redmond, WA
  Score: 0.70
  Relevant: True
  Complete: False
  Missing Info: ['One Microsoft Way', '98052']
  Reasoning: The extracted value 'Redmond, WA' is relevant as it refers to the location of the seller's address. However, it is incomplete because the full address in the source text is 'One Microsoft Way, Redmond, WA 98052'. The extracted value is missing the street address ('One 

# Test with Actual Data

- Correctness Evaluator
- Completeness Evaluator

In [7]:
import json

# Use relative path to the data file in the same directory structure
input_data_path = os.path.join("data", "labels.json")

with open(input_data_path, "r") as f:
    entity_extracted_values = json.load(f)

In [8]:
def extract_entities_per_page(pages):
    page_level_output = []

    for page in pages:
        page_number = page.get("page_number")
        entities = []

        entity_presence = page.get("entity_presence", {})
        entity_value = page.get("entity_value", {})
        chinese_entity_value = page.get("chinese_entity_value", {})

        for entity, present in entity_presence.items():
            if not present:
                continue

            # Prefer English, fallback to Chinese
            value = entity_value.get(entity) or chinese_entity_value.get(entity)

            if value:
                entities.append((entity, value))

        page_level_output.append({
            "page_number": page_number,
            "entities": entities
        })

    return page_level_output


# Example usage
entity_data = extract_entities_per_page(entity_extracted_values)


In [9]:
entity_data

[{'page_number': 1, 'entities': [('Applicant Name', 'Ms LOK WING CHING')]},
 {'page_number': 2, 'entities': [('Job Title of Applicant', 'DIRECTOR')]},
 {'page_number': 4,
  'entities': [('Business Registration Number of Employer', '21893829')]},
 {'page_number': 8,
  'entities': [('Height of Applicant', '174 cm'),
   ('Weight of Applicant', '77 kg')]}]

In [10]:
for page in entity_data:

    page_number = page["page_number"]
    entities = page["entities"]

    print(f"\n=== Page {page_number} ===")

    ocr_text_path = os.path.join("data", f"adi_text_page_{page_number}.txt")
    with open(ocr_text_path, "r") as f:
        ocr_text = f.read()


    for field_name, extracted_value in entities:
        result = correctness_eval.evaluate_field(
            field_name=field_name,
            extracted_value=extracted_value,
            source_text=ocr_text
        )
        
        status = "✓ CORRECT" if result.metadata['extraction_correct'] else "✗ INCORRECT"
        print(f"\nField: {field_name}")
        print(f"  Extracted Value: {extracted_value}")
        print(f"  Fuzzy Score: {result.metadata['fuzzy_score']:.2f}")
        print(f"  Status: {status}")
        if 'reason' in result.metadata:
            print(f"  Reason: {result.metadata['reason']}")


=== Page 1 ===

Field: Applicant Name
  Extracted Value: Ms LOK WING CHING
  Fuzzy Score: 1.00
  Status: ✓ CORRECT

=== Page 2 ===

Field: Job Title of Applicant
  Extracted Value: DIRECTOR
  Fuzzy Score: 1.00
  Status: ✓ CORRECT

=== Page 4 ===

Field: Business Registration Number of Employer
  Extracted Value: 21893829
  Fuzzy Score: 1.00
  Status: ✓ CORRECT

=== Page 8 ===

Field: Height of Applicant
  Extracted Value: 174 cm
  Fuzzy Score: 1.00
  Status: ✓ CORRECT

Field: Weight of Applicant
  Extracted Value: 77 kg
  Fuzzy Score: 1.00
  Status: ✓ CORRECT


# Test Evaluation Service

Test the unified `EvaluationService` that wraps both correctness and completeness evaluators.

In [11]:
for page in entity_data:

    page_number = page["page_number"]
    entities = page["entities"]

    print(f"\n=== Page {page_number} ===")

    ocr_text_path = os.path.join("data", f"adi_text_page_{page_number}.txt")
    with open(ocr_text_path, "r") as f:
        ocr_text = f.read()

    for field_name, extracted_value in entities:
        result = completeness_eval.evaluate_field(
            field_name=field_name,
            extracted_value=extracted_value,
            source_text=ocr_text 
        )

        print(f"\nField: {field_name}")
        print(f"  Extracted Value: {extracted_value}")
        print(f"  Score: {result.score:.2f}")
        print(f"  Relevant: {result.metadata['is_relevant']}")
        print(f"  Complete: {result.metadata['is_complete']}")
        print(f"  Missing Info: {result.metadata['missing_info']}")
        print(f"  Reasoning: {result.metadata['reasoning']}")


=== Page 1 ===

Field: Applicant Name
  Extracted Value: Ms LOK WING CHING
  Score: 1.00
  Relevant: True
  Complete: True
  Missing Info: []
  Reasoning: The extracted value 'Ms LOK WING CHING' matches the applicant's name as presented in the source text, which combines the title (Ms), family name (LOK), and given name (WING CHING). The Chinese name is present in the source but, per instructions, its absence does not affect completeness. No other relevant name information is missing.

=== Page 2 ===

Field: Job Title of Applicant
  Extracted Value: DIRECTOR
  Score: 1.00
  Relevant: True
  Complete: True
  Missing Info: []
  Reasoning: The extracted value 'DIRECTOR' matches the job title provided in the source text under 'Job Title (where applicable)職位(如適用)'. No additional details about the job title are present in the source text, and the Chinese translation is not required for completeness. Therefore, the extraction is both relevant and complete.

=== Page 4 ===

Field: Business Re

In [12]:
# Import the unified evaluation service
from evaluation_service import EvaluationService

# Initialize service with Azure OpenAI credentials (for completeness evaluation)
eval_service = EvaluationService(
    credential=DefaultAzureCredential(),
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_version=API_VERSION,
    deployment_name=DEPLOYMENT_NAME
)

print("✓ EvaluationService initialized")

✓ EvaluationService initialized


In [13]:
for page in entity_data:

    page_number = page["page_number"]
    entities = page["entities"]

    print(f"\n=== Page {page_number} ===")

    ocr_text_path = os.path.join("data", f"adi_text_page_{page_number}.txt")
    with open(ocr_text_path, "r") as f:
        ocr_text = f.read()

    for field_name, extracted_value in entities:
        result = eval_service.evaluate(
            field_name=field_name,
            extracted_value=extracted_value,
            source_text=ocr_text 
        )

        print(f"\nField: {field_name}")
        print(f"  Extracted Value: {extracted_value}")
        print(f"  Overall Score: {result['summary']['overall_score']:.3f}")
        
        # Correctness evaluation results
        if result['evaluations']['correctness']:
            print(f"  Correctness Score: {result['evaluations']['correctness']['score']:.3f}")
            print(f"  Extraction Correct: {result['evaluations']['correctness']['extraction_correct']}")
        
        # Completeness evaluation results
        if result['evaluations']['completeness']:
            print(f"  Completeness Score: {result['evaluations']['completeness']['score']:.3f}")
            print(f"  Relevant: {result['evaluations']['completeness']['is_relevant']}")
            print(f"  Complete: {result['evaluations']['completeness']['is_complete']}")
            print(f"  Missing Info: {result['evaluations']['completeness']['missing_info']}")
            print(f"  Reasoning: {result['evaluations']['completeness']['reasoning']}")

        print(f"\nSummary:")
        print(f"  Overall Score: {result['summary']['overall_score']:.3f}")
        print(f"  Evaluators: {result['summary']['evaluators_run']}")
        print(f"  Warnings: {result['summary']['warnings']}")
        print(f"  Errors: {result['summary']['errors']}")


=== Page 1 ===

Field: Applicant Name
  Extracted Value: Ms LOK WING CHING
  Overall Score: 1.000
  Correctness Score: 1.000
  Extraction Correct: True
  Completeness Score: 1.000
  Relevant: True
  Complete: True
  Missing Info: []
  Reasoning: The extracted value 'Ms LOK WING CHING' matches the applicant's name as presented in the source text, combining the title (Ms), family name (LOK), and given names (WING CHING). The Chinese name is present in the source but, per instructions, its omission does not affect completeness. No other relevant name information is missing.

Summary:
  Overall Score: 1.000
  Evaluators: ['correctness', 'completeness']
  Warnings: []
  Errors: []

=== Page 2 ===

Field: Job Title of Applicant
  Extracted Value: DIRECTOR
  Overall Score: 1.000
  Correctness Score: 1.000
  Extraction Correct: True
  Completeness Score: 1.000
  Relevant: True
  Complete: True
  Missing Info: []
  Reasoning: The extracted value 'DIRECTOR' matches the job title provided in the

In [15]:
entity_data

[{'page_number': 1, 'entities': [('Applicant Name', 'Ms LOK WING CHING')]},
 {'page_number': 2, 'entities': [('Job Title of Applicant', 'DIRECTOR')]},
 {'page_number': 4,
  'entities': [('Business Registration Number of Employer', '21893829')]},
 {'page_number': 8,
  'entities': [('Height of Applicant', '174 cm'),
   ('Weight of Applicant', '77 kg')]}]

In [16]:
# Test batch evaluation with actual data - both evaluators

for page in entity_data:
    page_number = page["page_number"]
    entities = page["entities"]

    print("\n" + "=" * 80)
    print(f"BATCH EVALUATION - PAGE {page_number}")
    print("=" * 80)

    # Convert entities to batch format expected by evaluate_batch
    batch_fields = [
        {"field_name": field_name, "value": extracted_value}
        for field_name, extracted_value in entities
    ]
    print(f"Batch fields prepared: {batch_fields}")

    # Load OCR text for this page
    ocr_text_path = os.path.join("data", f"adi_text_page_{page_number}.txt")
    with open(ocr_text_path, "r") as f:
        ocr_text = f.read()

    # Run batch evaluation with both evaluators (default)
    batch_results = eval_service.evaluate_batch(
        fields=batch_fields,
        source_text=ocr_text
    )

    print(f"\nTotal Fields: {batch_results['total_fields']}")

    # Display results in table format
    print(f"\n{'Field':<25} {'Overall':<12} {'Correct':<12} {'Complete':<12} {'Status':<15}")
    print("-" * 90)

    for r in batch_results['results']:
        overall = r['summary']['overall_score'] or 0.0
        
        # Correctness metrics
        corr_score = r['evaluations']['correctness']['score'] if r['evaluations']['correctness'] else 0.0
        is_correct = r['evaluations']['correctness'].get('extraction_correct', False) if r['evaluations']['correctness'] else False
        
        # Completeness metrics
        comp_score = r['evaluations']['completeness']['score'] if r['evaluations']['completeness'] else 0.0
        
        status = "✓ CORRECT" if is_correct else "✗ INCORRECT"
        print(f"{r['field_name']:<25} {overall:<12.3f} {corr_score:<12.3f} {comp_score:<12.3f} {status:<15}")

    # Display aggregate summary
    print(f"\n{'='*90}")
    print(f"AGGREGATE SUMMARY")
    print(f"{'='*90}")
    print(f"  Average Overall Score: {batch_results['aggregate_summary']['average_overall_score']:.3f}")
    print(f"  Average Correctness Score: {batch_results['aggregate_summary']['average_correctness_score']:.3f}")
    print(f"  Average Completeness Score: {batch_results['aggregate_summary'].get('average_completeness_score', 0.0):.3f}")
    print(f"  Fields Correct: {batch_results['aggregate_summary']['fields_correct']}/{batch_results['total_fields']}")
    print(f"  Failed Evaluations: {batch_results['aggregate_summary']['failed_evaluations']}")
    print(f"  Evaluators Run: {batch_results['aggregate_summary']['evaluators_run']}")


BATCH EVALUATION - PAGE 1
Batch fields prepared: [{'field_name': 'Applicant Name', 'value': 'Ms LOK WING CHING'}]

Total Fields: 1

Field                     Overall      Correct      Complete     Status         
------------------------------------------------------------------------------------------
Applicant Name            1.000        1.000        1.000        ✓ CORRECT      

AGGREGATE SUMMARY
  Average Overall Score: 1.000
  Average Correctness Score: 1.000
  Average Completeness Score: 1.000
  Fields Correct: 1/1
  Failed Evaluations: 0
  Evaluators Run: ['correctness', 'completeness']

BATCH EVALUATION - PAGE 2
Batch fields prepared: [{'field_name': 'Job Title of Applicant', 'value': 'DIRECTOR'}]

Total Fields: 1

Field                     Overall      Correct      Complete     Status         
------------------------------------------------------------------------------------------
Job Title of Applicant    1.000        1.000        1.000        ✓ CORRECT      

AGGREGATE S

In [17]:
for r in batch_results['results']:
    json_result = eval_service.to_json(r, indent=4)
    print(f"\nJSON Result for field '{r['field_name']}':\n{json_result}")


JSON Result for field 'Height of Applicant':
{
    "field_name": "Height of Applicant",
    "extracted_value": "174 cm",
    "evaluations": {
        "correctness": {
            "score": 1.0,
            "fuzzy_score": 1.0,
            "extraction_correct": true,
            "normalized_entity": "174 cm",
            "cleaned_source_length": 4535
        },
        "completeness": {
            "score": 1.0,
            "is_relevant": true,
            "is_complete": true,
            "missing_info": [],
            "reasoning": "The extracted value '174 cm' directly corresponds to the field name 'Height of Applicant' and matches the information provided in the source text. The source text lists height as '174 cm' and does not provide any additional details (such as feet/inches or other units) for this applicant. Therefore, the extraction is both relevant and complete."
        }
    },
    "summary": {
        "overall_score": 1.0,
        "evaluators_run": [
            "correctnes